# Diagnósticos de contextualidad — embeddings contextuales de turnos sobre Dialog2Flow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jumafernandez/doctorado-unsl/blob/main/contextual-turn-embeddings/notebooks/colab_d2f_contextuality_diagnostics.ipynb)

## Objetivo

El objetivo **no** es solo verificar que el modelo produce embeddings, sino comprobar si los
embeddings contextuales de turno son **realmente contextuales** de manera significativa. En
concreto, evaluamos si:

1. el embedding de un turno cambia cuando cambia su contexto de diálogo;
2. el contexto real ayuda más que un contexto mezclado o aleatorio;
3. el modelo no aplica solo una transformación "decorativa" sobre el embedding base;
4. utterances repetidas o genéricas se separan según el contexto;
5. los resultados de diagnóstico permiten guiar el siguiente paso de investigación.

> **Alcance / límites.** Este notebook es **solo de diagnóstico de contextualidad**. No
> implementa ANN, FAISS, MSS@10, jueces LLM ni tests estadísticos (Wilcoxon). Usa
> **únicamente embeddings precomputados** de Dialog2Flow (sin descargas de Hugging Face /
> SentenceTransformers). Por defecto **no entrena** ningún modelo: carga el modelo *smoke*
> entrenado en el notebook previo.

> **Advertencia importante.** Estos diagnósticos **no demuestran** superioridad en
> recuperación (retrieval) aguas abajo. Solo testean si el modelo usa el contexto del diálogo
> de forma medible y razonable. La validación final requerirá la evaluación posterior
> ANN/MSS cross-dialogue comparando las representaciones Static, Dynamic, EMA y Contextual.

## 2. Preparación del repositorio e instalación del paquete

Clonamos el repositorio (si no está presente) y entramos en el paquete. Para un repo privado,
cloná usando un token de acceso personal.

In [ ]:
import os

REPO_URL = "https://github.com/jumafernandez/doctorado-unsl"
REPO_DIR = "/content/doctorado-unsl"
PACKAGE_SUBDIR = "contextual-turn-embeddings"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repositorio ya presente en", REPO_DIR)

%cd {REPO_DIR}/{PACKAGE_SUBDIR}
!pwd

In [ ]:
# Instalación (modo editable). Solo embeddings precomputados: NO se necesitan
# transformers / sentence-transformers aquí.
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q matplotlib   # para los gráficos (ya viene en Colab)
print("Instalación completa.")

## 3. Montaje de Google Drive y definición de rutas

Todas las rutas son configurables. Cambiá `BASE_DIR` si tus archivos están en otra ubicación.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

# >>> Cambiá este directorio base si hace falta <<<
BASE_DIR = "/content/drive/MyDrive/Doctorado/Cursos/ANN/TF"

# Artefactos de entrada (Dialog2Flow, precomputados)
DIALOGS_PKL    = os.path.join(BASE_DIR, "dialogs-2.0.pkl")
IDS_NPY        = os.path.join(BASE_DIR, "ids.npy")
EMBEDDINGS_NPY = os.path.join(BASE_DIR, "embeddings_dialog2flow.npy")

# Modelo smoke entrenado por el notebook previo (colab_d2f_smoke.ipynb)
SMOKE_DIR       = os.path.join(BASE_DIR, "resultados", "contextual_turn_embeddings", "d2f_smoke")
SMOKE_MODEL_DIR = os.path.join(SMOKE_DIR, "model")
SMOKE_METADATA  = os.path.join(SMOKE_DIR, "metadata.csv")  # IDs de diálogos de entrenamiento

# Salida de los diagnósticos
DIAG_OUT = os.path.join(BASE_DIR, "resultados", "contextual_turn_embeddings",
                        "d2f_contextuality_diagnostics")

# Parámetros del diagnóstico (configurables)
N_DIAGNOSTIC_DIALOGUES = 500
MAX_TURNS = 64
SEED = 42
MIN_REPETITIONS = 10
ALLOW_TRAINING = False   # por defecto NO se entrena; solo diagnóstico

os.makedirs(DIAG_OUT, exist_ok=True)
print("Entradas:")
for p in [DIALOGS_PKL, IDS_NPY, EMBEDDINGS_NPY]:
    print(("  OK   " if os.path.exists(p) else "  FALTA "), p)
print("Modelo smoke esperado en:", SMOKE_MODEL_DIR)
print("Salida de diagnósticos  :", DIAG_OUT)

## 4. Carga de metadata, IDs y embeddings de Dialog2Flow

Cargamos el pickle de diálogos, los IDs y la matriz de embeddings precomputados, e imprimimos
diagnósticos de estructura para poder adaptar el armado del DataFrame si hiciera falta.

In [ ]:
import pickle
import numpy as np
import pandas as pd

with open(DIALOGS_PKL, "rb") as fh:
    dialogs = pickle.load(fh)

ids = np.load(IDS_NPY, allow_pickle=True)
embeddings = np.asarray(np.load(EMBEDDINGS_NPY), dtype=np.float32)

print("dialogs   :", type(dialogs))
if isinstance(dialogs, pd.DataFrame):
    print("  columnas:", list(dialogs.columns))
    display(dialogs.head(3))
elif isinstance(dialogs, dict):
    keys = list(dialogs.keys())
    print("  n claves:", len(keys), "| clave ejemplo:", keys[0])
    print("  tipo del valor:", type(dialogs[keys[0]]))
elif isinstance(dialogs, (list, tuple)):
    print("  longitud:", len(dialogs), "| tipo del item:", type(dialogs[0]))
    print("  ejemplo :", dialogs[0])

ids_arr = np.asarray(ids)
print("ids       :", ids_arr.shape, ids_arr.dtype, "| ejemplo:", ids_arr.ravel()[:4])
print("embeddings:", embeddings.shape, embeddings.dtype)

## 5. Verificaciones de alineación

Construimos el DataFrame canónico (`row_id, dialogue_id, turn_id, utterance` y `speaker` si
está disponible) manteniendo los embeddings **alineados por fila**. Si `ids.npy` codifica
`(dialogue_id, turn_id)`, alineamos por *join*; en caso contrario asumimos alineación
posicional. Después corremos verificaciones fuertes.

> ⚠️ Las exportaciones de Dialog2Flow varían. Revisá los diagnósticos de la celda anterior y
> ajustá los nombres de campos candidatos si la estructura difiere.

In [ ]:
# --- Adaptador robusto: aplana la metadata y alinea con los embeddings ---
SPEAKER_KEYS   = ("speaker", "role", "spk", "agent")
UTTERANCE_KEYS = ("utterance", "text", "utt", "content", "message")
TURNS_KEYS     = ("turns", "log", "utterances", "dialogue", "dialog")
DID_KEYS       = ("dialogue_id", "dialog_id", "id", "conv_id", "conversation_id")


def _first_key(d, keys, default=None):
    for k in keys:
        if k in d:
            return d[k]
    return default


def _coerce_turn(turn):
    if isinstance(turn, dict):
        return _first_key(turn, SPEAKER_KEYS), str(_first_key(turn, UTTERANCE_KEYS, ""))
    if isinstance(turn, (list, tuple)) and len(turn) >= 2:
        return turn[0], str(turn[1])
    return None, str(turn)


def flatten_dialogs(dialogs):
    if isinstance(dialogs, pd.DataFrame):
        df = dialogs.copy()
        ren = {}
        for cand in DID_KEYS:
            if cand in df.columns:
                ren[cand] = "dialogue_id"; break
        for cand in ("turn_id", "turn_idx", "turn", "index"):
            if cand in df.columns:
                ren[cand] = "turn_id"; break
        for cand in UTTERANCE_KEYS:
            if cand in df.columns:
                ren[cand] = "utterance"; break
        for cand in SPEAKER_KEYS:
            if cand in df.columns:
                ren[cand] = "speaker"; break
        df = df.rename(columns=ren)
        if "turn_id" not in df.columns:
            df["turn_id"] = df.groupby("dialogue_id").cumcount()
        return df
    rows = []
    items = dialogs.items() if isinstance(dialogs, dict) else enumerate(dialogs)
    for did, dlg in items:
        if isinstance(dlg, dict):
            real_id = _first_key(dlg, DID_KEYS, did)
            turns = _first_key(dlg, TURNS_KEYS, [])
        else:
            real_id, turns = did, dlg
        for t_idx, turn in enumerate(turns):
            spk, utt = _coerce_turn(turn)
            rows.append({"dialogue_id": real_id, "turn_id": t_idx,
                         "speaker": spk, "utterance": utt})
    return pd.DataFrame(rows)


def parse_ids_to_keys(ids):
    a = np.asarray(ids)
    if a.ndim == 2 and a.shape[1] >= 2:
        return pd.DataFrame({"dialogue_id": a[:, 0], "turn_id": a[:, 1]})
    if a.ndim == 1 and a.dtype.kind in ("U", "S", "O"):
        sample = str(a[0])
        for sep in ("__", "::", "_", "-", ":", "/"):
            if sep in sample:
                parts = [str(x).rsplit(sep, 1) for x in a]
                if all(len(p) == 2 for p in parts):
                    try:
                        tids = [int(p[1]) for p in parts]
                    except ValueError:
                        continue
                    return pd.DataFrame({"dialogue_id": [p[0] for p in parts], "turn_id": tids})
    return None


meta_df = flatten_dialogs(dialogs)
ids_keys = parse_ids_to_keys(ids)
if ids_keys is not None and len(ids_keys) == len(embeddings):
    lut = meta_df.copy()
    lut["dialogue_id"] = lut["dialogue_id"].astype(str)
    lut["turn_id"] = lut["turn_id"].astype(int)
    ids_keys["dialogue_id"] = ids_keys["dialogue_id"].astype(str)
    ids_keys["turn_id"] = ids_keys["turn_id"].astype(int)
    df = ids_keys.merge(lut, on=["dialogue_id", "turn_id"], how="left")
    alignment_method = "ids.npy (join)"
else:
    assert len(meta_df) == len(embeddings), (
        f"filas de metadata ({len(meta_df)}) != embeddings ({len(embeddings)}); "
        "adaptá esta celda a tu estructura de datos."
    )
    df = meta_df.reset_index(drop=True)
    alignment_method = "posicional"

if "utterance" not in df.columns:
    df["utterance"] = ""
df["utterance"] = df["utterance"].fillna("").astype(str)
if "speaker" in df.columns and not df["speaker"].notna().any():
    df = df.drop(columns=["speaker"])
df.insert(0, "row_id", np.arange(len(df), dtype=np.int64))
embeddings_aligned = embeddings
keep = [c for c in ["row_id", "dialogue_id", "turn_id", "utterance", "speaker"] if c in df.columns]
df = df[keep]
print("Método de alineación:", alignment_method)
df.head()

In [ ]:
# --- Verificaciones fuertes de alineación e integridad ---
assert len(df) == len(embeddings_aligned)
assert embeddings_aligned.shape[1] == 768, f"dim esperada 768, obtenida {embeddings_aligned.shape[1]}"
assert df["dialogue_id"].notna().all()
assert df["turn_id"].notna().all()
assert df["utterance"].notna().all()
assert df.duplicated(["dialogue_id", "turn_id"]).sum() == 0

n_dialogues = df["dialogue_id"].nunique()
n_turns = len(df)
print("Verificaciones OK.")
print("diálogos totales      :", n_dialogues)
print("turnos totales        :", n_turns)
print("shape de embeddings   :", embeddings_aligned.shape)
print("promedio turnos/diálogo:", round(n_turns / max(1, n_dialogues), 2))
print("alineación realizada por:", alignment_method)

## 6. Carga del modelo smoke entrenado

Cargamos el modelo entrenado en el notebook previo. **No se entrena** un modelo nuevo (a menos
que se configure `ALLOW_TRAINING`, no implementado aquí). Si no se encuentra el modelo, el
notebook se detiene con un mensaje claro.

In [ ]:
import torch
from contextual_turn_embeddings import ContextualTurnModel, get_device


def find_model_dir():
    for d in [SMOKE_MODEL_DIR, SMOKE_DIR]:
        has_cfg = os.path.exists(os.path.join(d, "config.json"))
        has_w = (os.path.exists(os.path.join(d, "model.safetensors"))
                 or os.path.exists(os.path.join(d, "pytorch_model.bin")))
        if has_cfg and has_w:
            return d
    return None


model_dir = find_model_dir()
if model_dir is None:
    raise FileNotFoundError(
        "No se encontró un modelo smoke entrenado en "
        f"'{SMOKE_MODEL_DIR}' ni en '{SMOKE_DIR}'. "
        "Ejecutá primero el notebook de entrenamiento smoke (colab_d2f_smoke.ipynb) "
        "para generar el modelo, o ajustá SMOKE_DIR/SMOKE_MODEL_DIR."
    )

device = get_device("auto")
model = ContextualTurnModel.from_pretrained(model_dir, device=str(device))
model.eval()
print("Modelo cargado desde:", model_dir)
print("device:", device, "| input_dim:", model.input_dim, "| output_dim:", model.output_dim,
      "| attention_mode:", model.config.attention_mode)

## 7. Selección del subconjunto diagnóstico

Seleccionamos diálogos **completos** (nunca turnos al azar). Si hay metadata del smoke,
excluimos sus diálogos de entrenamiento para evaluar sobre un conjunto *held-out*; si no, se
avisa que el subconjunto NO es held-out.

In [ ]:
held_out = False
train_dialogue_ids = set()
if os.path.exists(SMOKE_METADATA):
    train_dialogue_ids = set(pd.read_csv(SMOKE_METADATA)["dialogue_id"].astype(str).unique())
    held_out = True
    print("Diálogos de entrenamiento excluidos:", len(train_dialogue_ids))
else:
    print("[ADVERTENCIA] No se encontró metadata del smoke en:", SMOKE_METADATA)
    print("[ADVERTENCIA] Los diagnósticos se ejecutarán sobre un subconjunto NO held-out.")

df["_did_str"] = df["dialogue_id"].astype(str)
candidates = [d for d in pd.unique(df["_did_str"]) if d not in train_dialogue_ids]
selected = set(candidates[:N_DIAGNOSTIC_DIALOGUES])
mask = df["_did_str"].isin(selected).to_numpy()

df_subset = df[mask].drop(columns=["_did_str"]).reset_index(drop=True)
emb_subset = embeddings_aligned[mask]

print("diálogos seleccionados :", df_subset["dialogue_id"].nunique())
print("turnos seleccionados   :", len(df_subset))
print("promedio turnos/diálogo:", round(len(df_subset) / max(1, df_subset["dialogue_id"].nunique()), 2))
print("matriz de embeddings   :", emb_subset.shape)
print("held-out               :", held_out)

## 8. Construcción de funciones de corrupción de contexto

Definimos las funciones que arman, para cada turno objetivo, cuatro variantes de contexto:
`real`, `shuffled_within_dialogue`, `random_cross_dialogue` y `no_context`; más utilidades de
batch, métricas de reconstrucción y coseno. Luego precomputamos los embeddings base (`e_t`) y
contextuales con contexto real (`h_t`) de todo el subconjunto.

In [ ]:
import numpy as np
import torch


def l2norm(M, eps=1e-9):
    M = np.asarray(M, dtype=np.float32)
    return M / (np.linalg.norm(M, axis=1, keepdims=True) + eps)


def cosine_rows(A, B):
    return np.sum(l2norm(A) * l2norm(B), axis=1)


def mean_pairwise_cosine(M, max_n=200, rng=None):
    M = np.asarray(M, dtype=np.float32)
    if len(M) < 2:
        return np.nan
    if len(M) > max_n:
        rng = rng or np.random.default_rng(0)
        M = M[rng.choice(len(M), size=max_n, replace=False)]
    X = l2norm(M)
    S = X @ X.T
    k = len(M)
    return float((S.sum() - np.trace(S)) / (k * (k - 1)))


# --- Constructores de contexto (usan globals emb_subset, speaker_ids_all, use_speaker) ---
def _spk(positions):
    return speaker_ids_all[positions] if use_speaker else None


def make_real(positions, target_local):
    positions = list(positions)
    return {"emb": emb_subset[positions], "spk": _spk(positions), "target": int(target_local)}


def make_shuffled(positions, target_local, rng):
    positions = list(positions)
    perm = rng.permutation(len(positions))
    new_pos = [positions[p] for p in perm]
    new_target = int(np.where(perm == target_local)[0][0])
    return {"emb": emb_subset[new_pos], "spk": _spk(new_pos), "target": new_target}


def make_random(positions, target_local, rng, pool):
    positions = list(positions)
    L = len(positions)
    tgt = positions[target_local]
    size = min(L, len(pool))
    ctx = list(rng.choice(pool, size=size, replace=False))
    if len(ctx) <= target_local:
        ctx.append(tgt)
        tloc = len(ctx) - 1
    else:
        ctx[target_local] = tgt
        tloc = target_local
    return {"emb": emb_subset[ctx], "spk": _spk(ctx), "target": int(tloc)}


def make_no_context(positions, target_local):
    tgt = list(positions)[target_local]
    return {"emb": emb_subset[[tgt]], "spk": _spk([tgt]), "target": 0}


@torch.no_grad()
def run_batch(seqs, mask_target=False, batch_size=64):
    preds, tbases, hts = [], [], []
    for start in range(0, len(seqs), batch_size):
        chunk = seqs[start:start + batch_size]
        B = len(chunk)
        S = max(s["emb"].shape[0] for s in chunk)
        D = chunk[0]["emb"].shape[1]
        emb = torch.zeros(B, S, D, dtype=torch.float32)
        attn = torch.zeros(B, S, dtype=torch.long)
        use_spk = (model.speaker_embedding is not None) and all(s["spk"] is not None for s in chunk)
        spk = torch.zeros(B, S, dtype=torch.long) if use_spk else None
        tb = torch.zeros(B, D)
        tgt_idx = []
        for i, s in enumerate(chunk):
            L = s["emb"].shape[0]
            e = torch.from_numpy(np.asarray(s["emb"], dtype=np.float32)).float()
            tb[i] = e[s["target"]].clone()
            if mask_target:
                e = e.clone()
                e[s["target"]] = model.mask_embedding.detach().cpu().to(e.dtype)
            emb[i, :L] = e
            attn[i, :L] = 1
            if spk is not None:
                spk[i, :L] = torch.from_numpy(np.asarray(s["spk"])).long()
            tgt_idx.append(s["target"])
        emb = emb.to(device)
        attn = attn.to(device)
        spk_in = spk.to(device) if spk is not None else None
        h = model(emb, attn, spk_in)
        ar = torch.arange(B)
        h_t = h[ar, torch.tensor(tgt_idx)]
        if mask_target:
            pred = model.reconstruction_head(h_t)
            preds.append(pred.cpu().numpy())
            tbases.append(tb.numpy())
        else:
            hts.append(h_t.cpu().numpy())
    if mask_target:
        return np.vstack(preds), np.vstack(tbases)
    return np.vstack(hts)


def recon_metrics(pred, target):
    diff = pred - target
    mse = float(np.mean(diff ** 2))
    cosine_loss = float(np.mean(1.0 - cosine_rows(pred, target)))
    return mse, cosine_loss, mse + cosine_loss, len(pred)


print("Funciones de diagnóstico definidas.")

In [ ]:
# --- Precomputo: e_t (base) y h_t (contextual, contexto real) de todo el subconjunto ---
from contextual_turn_embeddings import encode_dialogues
from contextual_turn_embeddings.data import build_speaker_map, encode_speakers

use_speaker = ((model.speaker_embedding is not None)
               and ("speaker" in df_subset.columns)
               and df_subset["speaker"].notna().any())
if use_speaker:
    speaker_map = build_speaker_map(df_subset, model.config.num_speakers, None)
    speaker_ids_all = encode_speakers(df_subset["speaker"].to_list(), speaker_map,
                                      model.config.num_speakers)
else:
    speaker_ids_all = None
print("usa speaker embeddings:", use_speaker)

e_base = np.asarray(emb_subset, dtype=np.float32)
_matrix, _meta = encode_dialogues(model, df_subset, embeddings=emb_subset, device=str(device))
_h_by_rowid = {int(r): _matrix[i] for i, r in enumerate(_meta["row_id"].to_numpy())}
H_real = np.vstack([_h_by_rowid[int(r)] for r in df_subset["row_id"].to_numpy()]).astype(np.float32)
print("e_base:", e_base.shape, "| H_real:", H_real.shape)

# Índice de diálogos -> posiciones (en df_subset/emb_subset), truncado a MAX_TURNS.
dialogue_index = {}
for did, grp in df_subset.groupby("dialogue_id", sort=False):
    positions = grp.index.to_numpy()
    if len(positions) > MAX_TURNS:
        positions = positions[:MAX_TURNS]
    dialogue_index[did] = positions
pool_all = np.arange(len(df_subset))
did_arr = df_subset["dialogue_id"].to_numpy()
print("diálogos indexados:", len(dialogue_index))

## 9. Diagnóstico 1: reconstrucción enmascarada con contexto real vs contexto alterado

**El diagnóstico más importante.** Para cada diálogo elegimos un turno objetivo (turno medio),
lo **enmascaramos** y pedimos al modelo reconstruir su embedding base a partir del contexto
disponible, bajo cuatro condiciones: contexto real, mezclado dentro del diálogo, aleatorio de
otros diálogos, y sin contexto.

**Patrón esperado:** `loss(real) < loss(shuffled)`, `loss(real) < loss(random)`,
`loss(real) < loss(no_context)`. Si el contexto real no es mejor, el modelo podría no estar
usando la estructura conversacional de forma significativa.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEED)
conditions = ["real_context", "shuffled_within_dialogue",
              "random_cross_dialogue_context", "no_context"]
seqs_by_cond = {c: [] for c in conditions}
for did, positions in dialogue_index.items():
    if len(positions) < 2:
        continue
    tl = len(positions) // 2
    pool = pool_all[did_arr[pool_all] != did]
    seqs_by_cond["real_context"].append(make_real(positions, tl))
    seqs_by_cond["shuffled_within_dialogue"].append(make_shuffled(positions, tl, rng))
    seqs_by_cond["random_cross_dialogue_context"].append(make_random(positions, tl, rng, pool))
    seqs_by_cond["no_context"].append(make_no_context(positions, tl))

rows = []
for c in conditions:
    pred, tgt = run_batch(seqs_by_cond[c], mask_target=True)
    mse, cl, tot, n = recon_metrics(pred, tgt)
    rows.append({"condition": c, "mse": mse, "cosine_loss": cl, "total_loss": tot, "n_positions": n})
loss_df = pd.DataFrame(rows)
print(loss_df.to_string(index=False))
loss_df.to_csv(os.path.join(DIAG_OUT, "diagnostic_losses_by_context.csv"), index=False)

plt.figure(figsize=(7, 4))
plt.bar(loss_df["condition"], loss_df["total_loss"])
plt.ylabel("total_loss (MSE + coseno)")
plt.title("Pérdida de reconstrucción por condición de contexto")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(DIAG_OUT, "loss_by_context_condition.png"), dpi=120)
plt.show()

real = loss_df.loc[loss_df.condition == "real_context", "total_loss"].iloc[0]
others = loss_df.loc[loss_df.condition != "real_context", "total_loss"]
real_context_best = bool((real < others).all())
print("")
if real_context_best:
    print("[OK] El contexto real produce la menor pérdida de reconstrucción.")
else:
    print("[ADVERTENCIA] El contexto real NO produjo la menor pérdida. El modelo podría no "
          "estar usando la estructura conversacional de forma significativa.")

## 10. Diagnóstico 2: sensibilidad del embedding contextual al contexto

Para los mismos turnos objetivo (sin enmascarar) calculamos su embedding contextual bajo
contexto real, mezclado, aleatorio y sin contexto, y medimos `cos(h_real, h_*)` y
`cos(e_base, h_real)`.

**Interpretación:** si todas las similitudes están cerca de 1, el modelo sería casi
insensible al contexto; si son muy bajas, podría estar sobre-transformando o desestabilizando
el espacio; un modelo contextual útil muestra cambios **medibles pero no caóticos**.

In [ ]:
rng = np.random.default_rng(SEED)
target_rows, seqs_sh, seqs_rd, seqs_nc = [], [], [], []
for did, positions in dialogue_index.items():
    if len(positions) < 2:
        continue
    tl = len(positions) // 2
    pool = pool_all[did_arr[pool_all] != did]
    target_rows.append(positions[tl])
    seqs_sh.append(make_shuffled(positions, tl, rng))
    seqs_rd.append(make_random(positions, tl, rng, pool))
    seqs_nc.append(make_no_context(positions, tl))
target_rows = np.array(target_rows)

h_real = H_real[target_rows]
e_base_t = e_base[target_rows]
h_sh = run_batch(seqs_sh)
h_rd = run_batch(seqs_rd)
h_nc = run_batch(seqs_nc)

cos_sh = cosine_rows(h_real, h_sh)
cos_rd = cosine_rows(h_real, h_rd)
cos_nc = cosine_rows(h_real, h_nc)
cos_eb = cosine_rows(e_base_t, h_real)


def _summ(name, x):
    return {"comparison": name, "mean": float(np.mean(x)), "median": float(np.median(x)),
            "std": float(np.std(x)), "p10": float(np.percentile(x, 10)),
            "p90": float(np.percentile(x, 90))}


sens = pd.DataFrame([
    _summ("cos(h_real,h_shuffled)", cos_sh),
    _summ("cos(h_real,h_random)", cos_rd),
    _summ("cos(h_real,h_no_context)", cos_nc),
    _summ("cos(e_base,h_real)", cos_eb),
])
print(sens.to_string(index=False))
sens.to_csv(os.path.join(DIAG_OUT, "context_sensitivity_summary.csv"), index=False)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (name, data) in zip(axes.ravel(), [
        ("cos(h_real,h_shuffled)", cos_sh), ("cos(h_real,h_random)", cos_rd),
        ("cos(h_real,h_no_context)", cos_nc), ("cos(e_base,h_real)", cos_eb)]):
    ax.hist(data, bins=30)
    ax.set_title(name)
fig.suptitle("Sensibilidad al contexto (similitud coseno)")
fig.tight_layout()
fig.savefig(os.path.join(DIAG_OUT, "context_sensitivity_histograms.png"), dpi=120)
plt.show()

ctx_changes = bool((np.mean(cos_sh) < 0.999) or (np.mean(cos_rd) < 0.999))
print("")
print("¿Los embeddings contextuales cambian con el contexto?", ctx_changes)

## 11. Diagnóstico 3: misma utterance, distinto contexto

Buscamos utterances repetidas (normalizadas) que aparezcan en al menos `MIN_REPETITIONS`
diálogos (típicamente genéricas: "yes", "no", "thank you", "okay"...). Comparamos la
similitud media par-a-par entre sus embeddings **base** vs **contextuales**.

**Esperado:** mismo texto + contexto distinto → los embeddings contextuales se **separan más**
(dispersión contextual = sim_base − sim_contextual > 0).

In [ ]:
import re


def normalize_utt(u):
    return re.sub(r"\s+", " ", str(u).strip().lower())


df_subset = df_subset.copy()
df_subset["utt_norm"] = df_subset["utterance"].map(normalize_utt)
counts = df_subset["utt_norm"].value_counts()
repeated = counts[counts >= MIN_REPETITIONS].index.tolist()
rng = np.random.default_rng(SEED)

rows = []
utt_norm_arr = df_subset["utt_norm"].to_numpy()
for utt in repeated:
    idx = np.where(utt_norm_arr == utt)[0]
    base_sim = mean_pairwise_cosine(e_base[idx], max_n=200, rng=rng)
    ctx_sim = mean_pairwise_cosine(H_real[idx], max_n=200, rng=rng)
    disp = (base_sim - ctx_sim) if (base_sim == base_sim and ctx_sim == ctx_sim) else np.nan
    rows.append({"utterance": utt, "n_occurrences": int(len(idx)),
                 "base_pairwise_cos": base_sim, "contextual_pairwise_cos": ctx_sim,
                 "contextual_dispersion": disp})

disp_df = pd.DataFrame(rows).sort_values("contextual_dispersion", ascending=False)
disp_df.to_csv(os.path.join(DIAG_OUT, "repeated_utterance_contextual_dispersion.csv"), index=False)

if len(disp_df) == 0:
    generic_separated = False
    print("[ADVERTENCIA] No se encontraron utterances repetidas (>= %d) en el subconjunto. "
          "Probá aumentar N_DIAGNOSTIC_DIALOGUES o bajar MIN_REPETITIONS." % MIN_REPETITIONS)
else:
    generic_separated = bool(np.nanmedian(disp_df["contextual_dispersion"]) > 0)
    print("Utterances repetidas (>= %d):" % MIN_REPETITIONS, len(disp_df))
    print("Top por dispersión contextual:")
    print(disp_df.head(15).to_string(index=False))
    print("")
    print("¿Las utterances genéricas se separan según el contexto (mediana>0)?", generic_separated)

## 12. Diagnóstico 4: inspección cualitativa de vecinos exactos

**Sin FAISS ni ANN.** Solo coseno exacto (producto matricial) sobre el subconjunto pequeño.
Para unas pocas consultas comparamos los vecinos recuperados por el embedding **base** `e_t` vs
el **contextual** `h_t`, evitando vecinos del mismo diálogo para enfocar la similitud
cross-dialogue. Es solo una inspección cualitativa, **no** la evaluación final ANN/MSS.

In [ ]:
N_QUERIES = 5
TOPK = 5
base_norm = l2norm(e_base)
ctx_norm = l2norm(H_real)
did_arr = df_subset["dialogue_id"].to_numpy()
turn_arr = df_subset["turn_id"].to_numpy()
utt_arr = df_subset["utterance"].to_numpy()
rng = np.random.default_rng(SEED)
query_idx = rng.choice(len(df_subset), size=min(N_QUERIES, len(df_subset)), replace=False)


def topk_neighbors(qi, M_norm, k, exclude_same_dialogue=True):
    sims = M_norm @ M_norm[qi]
    sims[qi] = -np.inf
    if exclude_same_dialogue:
        sims[did_arr == did_arr[qi]] = -np.inf
    order = np.argsort(-sims)[:k]
    return [(int(j), float(sims[j])) for j in order]


records = []
for qi in query_idx:
    print("=" * 90)
    print("QUERY  | dlg=%s turn=%s | %s" % (did_arr[qi], turn_arr[qi], utt_arr[qi]))
    for space, M_norm in [("base", base_norm), ("contextual", ctx_norm)]:
        print("  [%s]" % space)
        for rank, (j, c) in enumerate(topk_neighbors(qi, M_norm, TOPK), start=1):
            print("    #%d cos=%.3f | dlg=%s turn=%s | %s" %
                  (rank, c, did_arr[j], turn_arr[j], utt_arr[j]))
            records.append({"query_dialogue_id": did_arr[qi], "query_turn_id": int(turn_arr[qi]),
                            "query_utterance": utt_arr[qi], "space": space, "rank": rank,
                            "neighbor_dialogue_id": did_arr[j], "neighbor_turn_id": int(turn_arr[j]),
                            "neighbor_utterance": utt_arr[j], "cosine": c})

neigh_df = pd.DataFrame(records)
neigh_df.to_csv(os.path.join(DIAG_OUT, "qualitative_neighbors_base_vs_contextual.csv"), index=False)
print("")
print("Guardado:", os.path.join(DIAG_OUT, "qualitative_neighbors_base_vs_contextual.csv"))

## 13. Diagnóstico 5: resumen del cambio geométrico

Calculamos `cos(e_t, h_t)`, las normas de `e_t` y `h_t`, y el solapamiento de vecinos exactos
`Overlap@10(base, contextual)` sobre una muestra del subconjunto (coseno exacto únicamente).

**Interpretación:** overlap ≈ 1 → el modelo casi no cambia las vecindades; overlap ≈ 0 →
podría destruir la geometría base; un valor intermedio sugiere reorganización contextual
significativa.

In [ ]:
cos_eh = cosine_rows(e_base, H_real)
norm_e = np.linalg.norm(e_base, axis=1)
norm_h = np.linalg.norm(H_real, axis=1)


def _g(name, x):
    return {"metric": name, "mean": float(np.mean(x)), "median": float(np.median(x)),
            "std": float(np.std(x)), "p10": float(np.percentile(x, 10)),
            "p90": float(np.percentile(x, 90))}


geo = pd.DataFrame([_g("cos(e_t,h_t)", cos_eh), _g("norm(e_t)", norm_e), _g("norm(h_t)", norm_h)])

K = 10
Q = min(500, len(df_subset))
rng = np.random.default_rng(SEED)
qidx = rng.choice(len(df_subset), size=Q, replace=False)
base_norm = l2norm(e_base)
ctx_norm = l2norm(H_real)
overlaps = []
for qi in qidx:
    sb = base_norm @ base_norm[qi]; sb[qi] = -np.inf
    sc = ctx_norm @ ctx_norm[qi]; sc[qi] = -np.inf
    tb = set(np.argsort(-sb)[:K].tolist())
    tc = set(np.argsort(-sc)[:K].tolist())
    overlaps.append(len(tb & tc) / K)
overlap10 = float(np.mean(overlaps))

geo_out = pd.concat([geo, pd.DataFrame([{"metric": "overlap@10_base_vs_contextual",
                                         "mean": overlap10}])], ignore_index=True)
print(geo_out.to_string(index=False))
print("Overlap@10(base, contextual):", round(overlap10, 4))
geo_out.to_csv(os.path.join(DIAG_OUT, "geometry_shift_summary.csv"), index=False)

plt.figure(figsize=(7, 4))
plt.hist(cos_eh, bins=30)
plt.xlabel("cos(e_t, h_t)")
plt.title("Cambio geométrico base vs contextual")
plt.tight_layout()
plt.savefig(os.path.join(DIAG_OUT, "geometry_shift_histogram.png"), dpi=120)
plt.show()

cos_eh_mean = float(np.mean(cos_eh))
not_chaotic = bool(0.2 <= cos_eh_mean <= 0.98)
print("media cos(e_t,h_t):", round(cos_eh_mean, 4), "| ¿cambio no caótico?", not_chaotic)

## 14. Guardado de salidas

Guardamos la configuración del diagnóstico y verificamos que todos los artefactos esperados
existan en el directorio de salida.

In [ ]:
import json

diag_config = {
    "BASE_DIR": BASE_DIR,
    "DIAG_OUT": DIAG_OUT,
    "SMOKE_MODEL_DIR": model_dir,
    "alignment_method": alignment_method,
    "held_out": bool(held_out),
    "N_DIAGNOSTIC_DIALOGUES": N_DIAGNOSTIC_DIALOGUES,
    "MAX_TURNS": MAX_TURNS,
    "SEED": SEED,
    "MIN_REPETITIONS": MIN_REPETITIONS,
    "n_dialogues_subset": int(df_subset["dialogue_id"].nunique()),
    "n_turns_subset": int(len(df_subset)),
    "embedding_dim": int(e_base.shape[1]),
    "attention_mode": model.config.attention_mode,
    "use_speaker_embeddings": bool(use_speaker),
}
with open(os.path.join(DIAG_OUT, "diagnostic_config.json"), "w", encoding="utf-8") as fh:
    json.dump(diag_config, fh, indent=2, ensure_ascii=False)

expected_files = [
    "diagnostic_losses_by_context.csv",
    "context_sensitivity_summary.csv",
    "repeated_utterance_contextual_dispersion.csv",
    "qualitative_neighbors_base_vs_contextual.csv",
    "geometry_shift_summary.csv",
    "diagnostic_config.json",
    "loss_by_context_condition.png",
    "context_sensitivity_histograms.png",
    "geometry_shift_histogram.png",
]
print("Artefactos en", DIAG_OUT, ":")
all_saved = True
for f in expected_files:
    ok = os.path.exists(os.path.join(DIAG_OUT, f))
    all_saved = all_saved and ok
    print(("  OK   " if ok else "  FALTA "), f)
saved_ok = bool(all_saved)

## 15. Checklist final de interpretación

Resumen interpretativo. Recordá: estos diagnósticos **no** prueban superioridad en
recuperación aguas abajo; solo testean si el modelo usa el contexto del diálogo de forma
medible y razonable. La validación final requerirá la evaluación posterior ANN/MSS
cross-dialogue contra las representaciones Static, Dynamic, EMA y Contextual.

In [ ]:
no_nans = bool((not np.isnan(H_real).any()) and (not np.isnan(e_base).any()))


def box(b):
    return "[x]" if b else "[ ]"


print("CHECKLIST DE INTERPRETACIÓN")
print(box(real_context_best),
      "el contexto real produce menor pérdida de reconstrucción que el contexto "
      "mezclado/aleatorio/sin contexto")
print(box(ctx_changes), "los embeddings contextuales cambian cuando cambia el contexto")
print(box(not_chaotic),
      "los cambios no son caóticos: cos(e_t, h_t) se mantiene en un rango razonable "
      "(media=%.3f)" % cos_eh_mean)
print(box(generic_separated),
      "utterances genéricas repetidas se separan de acuerdo con el contexto")
print("[ ] la inspección de vecinos exactos sugiere menos repetición lexical y más "
      "similitud funcional (revisar manualmente arriba)")
print(box(no_nans), "no hay NaNs")
print(box(saved_ok), "los diagnósticos se guardaron correctamente")

print("")
print("NOTA: estos diagnósticos NO prueban superioridad en retrieval. Solo testean si el "
      "modelo usa el contexto de forma medible. La validación final requiere la evaluación "
      "ANN/MSS cross-dialogue (Static, Dynamic, EMA, Contextual), fuera del alcance de este "
      "notebook.")